<a href="https://colab.research.google.com/github/hamzaali-712/ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
import numpy as np

# Dummy data for demonstration purposes to make df available and avoid KeyError
data = {
    "search_volume": [100, 200, np.nan, 150],
    "competition": [0.5, 0.7, 0.3, 0.6],
    "cpc": [1.0, 1.2, 0.8, 1.1],
    "word_count": [500, 600, np.nan, 550],
    "char_count": [3000, 4000, 2500, 3500],
    "impressions_90d": [1000, 2000, 1500, 1200],
    "clicks_90d": [100, 200, 150, 120],
    "pageviews_90d": [500, 1000, 750, 600],
    "sessions_90d": [400, 800, 600, 480],
    "ai_sessions_90d": [50, 100, 75, 60],
    "scroll_events_90d": [200, 400, 300, 240],
    "impressions_last_30d": [300, 700, 500, 360],
    "clicks_last_30d": [30, 70, 50, 36],
    "content_age_days": [10, 20, np.nan, 15],
    "content_type": ["blog", "news", "blog", "guide"],
    "competition_level": ["High", "Medium", "Low", "High"]
}
df = pd.DataFrame(data)

# Numeric features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "content_age_days"
]

# Fill missing values
for col in numeric_features:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Categorical features
categorical_features = [
    "content_type",
    "competition_level"
]

for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

X = pd.get_dummies(
    df[numeric_features + categorical_features],
    drop_first=True
)

print(X.shape)
X.head()

(4, 18)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,ai_sessions_90d,scroll_events_90d,impressions_last_30d,clicks_last_30d,content_age_days,content_type_guide,content_type_news,competition_level_Low,competition_level_Medium
0,100.0,0.5,1.0,500.0,3000,1000,100,500,400,50,200,300,30,10.0,False,False,False,False
1,200.0,0.7,1.2,600.0,4000,2000,200,1000,800,100,400,700,70,20.0,False,True,False,True
2,150.0,0.3,0.8,550.0,2500,1500,150,750,600,75,300,500,50,15.0,False,False,True,False
3,150.0,0.6,1.1,550.0,3500,1200,120,600,480,60,240,360,36,15.0,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
feature_notes = pd.DataFrame({
    "Feature":[
        "search_volume",
        "competition",
        "word_count",
        "content_age_days",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "content_type",
        "competition_level"
    ],
    "Meaning":[
        "Estimated keyword demand",
        "Keyword competition",
        "Length of content",
        "Age of page",
        "Last 90-day impressions",
        "Last 90-day clicks",
        "Last 90-day sessions",
        "Content category",
        "Competition bucket"
    ],
    "Missing Handling":[
        "Median",
        "Median",
        "Median",
        "Median",
        "Median",
        "Median",
        "Median",
        "Unknown",
        "Unknown"
    ],
    "Available Before Prediction":[
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

feature_notes

,Feature,Meaning,Missing Handling,Available Before Prediction
0,search_volume,Estimated keyword demand,Median,Yes
1,competition,Keyword competition,Median,Yes
2,word_count,Length of content,Median,Yes
3,content_age_days,Age of page,Median,Yes
4,impressions_90d,Last 90-day impressions,Median,Yes
5,clicks_90d,Last 90-day clicks,Median,Yes
6,sessions_90d,Last 90-day sessions,Median,Yes
7,content_type,Content category,Unknown,Yes
8,competition_level,Competition bucket,Unknown,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
suspected_leakage = [
    "trend_direction",
    "impressions_next_30d",
    "clicks_next_30d",
    "future_pageviews",
    "refresh_label",
    "priority_score",
    "manual_decision"
]

existing = [c for c in suspected_leakage if c in df.columns]

print("Potential leakage columns found:")
print(existing)

assert "trend_direction" not in X.columns

print("Leakage check passed.")

Potential leakage columns found:
[]
Leakage check passed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
excluded = pd.DataFrame({
    "Excluded Column":[
        "content_id",
        "client_id",
        "provider_used",
        "model_used",
        "trend_direction"
    ],
    "Reason":[
        "Identifier only",
        "Identifier only",
        "Operational metadata",
        "Operational metadata",
        "Label / target leakage"
    ]
})

excluded

,Excluded Column,Reason
0,content_id,Identifier only
1,client_id,Identifier only
2,provider_used,Operational metadata
3,model_used,Operational metadata
4,trend_direction,Label / target leakage


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.